# Deep Learning, Hands-on: rock, paper, scissors

The page with the ideas and activities: **https://gradlabs.in/learn/deep-learning/**

1. **PyTorch basics:** teach one neuron to fit some dots.
2. **Your first network:** a CNN that tells rock, paper and scissors apart.
3. **Borrow a trained network:** fine-tune one with just 60 photos, then the 🏆 leaderboard.
4. **Play against it:** an app that reads your hand.

**How to use it:** click a cell, press **Shift + Enter** to run it, and go from top to bottom.

## 0 · Setup: run this first

Turn on the free GPU first: **Runtime → Change runtime type → T4 GPU → Save**.

If Colab disconnects or you fall behind, run this cell again, then jump to the start of any part. Each part sets itself up.

In [ ]:
import os, zipfile, urllib.request
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image

print("PyTorch", torch.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print("✓ GPU found:", torch.cuda.get_device_name(0))
else:
    print("No GPU. It still works, just slower.")
    print("To turn it on: Runtime → Change runtime type → T4 GPU → Save, then run this cell again.")

# ---- Helpers for Parts 2–4. You don't need to read these to follow along. ----
GESTURES = ["paper", "rock", "scissors"]
SIZE = 128                     # every photo is shrunk to 128 × 128 pixels
DATA = "https://storage.googleapis.com/download.tensorflow.org/data/"

def _folder_with_gestures(top):
    for here, dirs, _ in os.walk(top):
        if all(g in dirs for g in GESTURES):
            return here
    raise FileNotFoundError(f"No rock/paper/scissors folders in {top}")

def _load(top):
    folder = _folder_with_gestures(top)
    pictures, labels = [], []
    for label, g in enumerate(GESTURES):
        for name in sorted(os.listdir(os.path.join(folder, g))):
            img = Image.open(os.path.join(folder, g, name)).convert("RGB").resize((SIZE, SIZE))
            pictures.append(torch.from_numpy(np.array(img)).permute(2, 0, 1))
            labels.append(label)
    return torch.stack(pictures), torch.tensor(labels)

def get_hand_photos():
    """Downloads the photos once (about 30 seconds), then loads them as tensors."""
    for name in ["rps", "rps-test-set"]:
        if not os.path.isdir(name):
            urllib.request.urlretrieve(DATA + name + ".zip", name + ".zip")
            with zipfile.ZipFile(name + ".zip") as z:
                z.extractall(name)
    return _load("rps"), _load("rps-test-set")

def pick(x, y, per_gesture, seed=0):
    """The same few photos of each gesture for everyone, so the leaderboard is fair."""
    g = torch.Generator().manual_seed(seed)
    keep = torch.cat([torch.nonzero(y == k).flatten()[torch.randperm(int((y == k).sum()), generator=g)[:per_gesture]] for k in range(3)])
    return x[keep], y[keep]

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def prepare(pictures, size, flip=False):
    """Photos → numbers the network can use: scaled, optionally flipped, resized."""
    x = pictures.to(device).float() / 255
    if flip:
        mirror = torch.rand(len(x), device=device) < 0.5
        x[mirror] = x[mirror].flip(3)
    if x.shape[-1] != size:
        x = F.interpolate(x, size=(size, size), mode="bilinear", antialias=True, align_corners=False)
    return (x - MEAN.to(device)) / STD.to(device)

def batches(x, y, size, shuffle=False):
    order = torch.randperm(len(x)) if shuffle else torch.arange(len(x))
    for i in range(0, len(x), size):
        part = order[i:i + size]
        yield x[part], y[part]

def accuracy(model, x, y, size):
    model.eval()
    right = 0
    with torch.no_grad():
        for xb, yb in batches(x, y, 128):
            right += (model(prepare(xb, size)).argmax(1).cpu() == yb).sum().item()
    return 100 * right / len(x)

def train(model, x, y, rounds, lr=0.001, size=64, flip=False, batch=32):
    """The training loop from the page, run for a few rounds. Prints the test score after each."""
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for r in range(1, rounds + 1):
        model.train()
        for m in model.modules():   # frozen layers keep what they learned, including their averages
            if isinstance(m, nn.modules.batchnorm._BatchNorm) and not any(p.requires_grad for p in m.parameters()):
                m.eval()
        for xb, yb in batches(x, y, batch, shuffle=True):
            guesses = model(prepare(xb, size, flip))    # 1 · guess
            loss = loss_fn(guesses, yb.to(device))      # 2 · how wrong
            optimizer.zero_grad()                       # 3 · clear old gradients
            loss.backward()                             # 4 · gradients
            optimizer.step()                            # 5 · step downhill
        print(f"round {r}: test {accuracy(model, test_x, test_y, size):.1f}%")
    return model

def show(pictures, titles, colours=None, n=8):
    n = min(n, len(pictures))
    fig, axes = plt.subplots(1, n, figsize=(2 * n, 2.4))
    for i, ax in enumerate(np.atleast_1d(axes)):
        ax.imshow(pictures[i].permute(1, 2, 0)); ax.axis("off")
        ax.set_title(titles[i], color=(colours[i] if colours else "black"), fontsize=10)
    plt.show()
print("Setup done.")

## Part 1 · PyTorch basics

The same dots as on the page: hours studied → test score. A **tensor** is PyTorch's list of numbers: maths on it happens to every number at once.

In [ ]:
hours  = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
scores = torch.tensor([3.0, 5.0, 7.0, 9.0, 11.0])

weight, bias = 0.5, 1.0
guesses = weight * hours + bias                      # output = weight × input + bias
loss = ((guesses - scores) ** 2).mean()              # loss = average of (guess − real)²
print("guesses:", guesses)
print("loss:", round(loss.item(), 2))

✏️ Change `weight` and `bias`, run the cell again, and get the loss to 0.

**Let PyTorch find the gradient.** `requires_grad=True` tells PyTorch to track this number, and `loss.backward()` works out its gradient. From weight 0 and bias 0 you should see −50 and −14, the same as on the page.

In [ ]:
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
loss = ((w * hours + b - scores) ** 2).mean()
loss.backward()
print("gradient for weight:", round(w.grad.item(), 2))
print("gradient for bias:  ", round(b.grad.item(), 2))

**Walk downhill.** `new weight = weight − learning rate × gradient`, 100 times. `torch.no_grad()` means "this step isn't part of the loss", and `.grad.zero_()` clears the old gradient, because PyTorch adds new ones on top.

In [ ]:
learning_rate = 0.05     # ✏️ try 0.1, then 0.01
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

for step in range(1, 101):
    loss = ((w * hours + b - scores) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
    w.grad.zero_(); b.grad.zero_()
    if step in (1, 2, 5, 10, 50, 100):
        print(f"step {step:3}: weight {w.item():.3f}, bias {b.item():.3f}, loss {loss.item():.4f}")

**The way real code does it:** `nn.Linear` is a neuron with its own weight and bias, and an **optimizer** takes the step downhill for every weight at once. The five lines from the page:

In [ ]:
model = nn.Linear(1, 1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
X, Y = hours.unsqueeze(1), scores.unsqueeze(1)        # one row per example

for step in range(500):
    guesses = model(X)                  # 1 · guess
    loss = loss_fn(guesses, Y)          # 2 · how wrong
    optimizer.zero_grad()               # 3 · clear old gradients
    loss.backward()                     # 4 · gradients
    optimizer.step()                    # 5 · step downhill

print("weight:", round(model.weight.item(), 2), " bias:", round(model.bias.item(), 2))

Weight 2, bias 1: found by walking downhill. ✅ **Back to the page** for layers and convolution.

## Part 2 · Your first network: rock, paper, scissors

About 2,500 photos of hands to learn from, and 372 more kept aside as the **test set**. They are computer-drawn hands on white backgrounds, made for learning.

In [ ]:
(train_x, train_y), (test_x, test_y) = get_hand_photos()
print(len(train_x), "photos to learn from,", len(test_x), "test photos")
every = torch.arange(0, len(train_x), len(train_x) // 8)[:8]
show(train_x[every], [GESTURES[k] for k in train_y[every]])

**A small CNN.** Each `Conv2d` is a set of filters that slide over the photo, like the one on the page. `MaxPool2d(2)` halves the picture, keeping the biggest number in each 2 × 2 square, so later filters see a bigger area. At the end: 3 scores, one per gesture.

To keep it quick, this network sees the photos at 64 × 64.

In [ ]:
def make_cnn(width=32):
    return nn.Sequential(
        nn.Conv2d(3, width, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),               # 64 → 32
        nn.Conv2d(width, 2 * width, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),       # 32 → 16
        nn.Conv2d(2 * width, 4 * width, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 16 → 8
        nn.Flatten(), nn.Linear(4 * width * 8 * 8, 128), nn.ReLU(), nn.Linear(128, 3),
    ).to(device)

cnn = train(make_cnn(), train_x, train_y, rounds=4, size=64)    # about 1 minute on a T4 GPU
scratch_all = accuracy(cnn, test_x, test_y, 64)
print(f"Trained on all {len(train_x)} photos: {scratch_all:.1f}% right on the test set")

Where does it go wrong? Red titles are mistakes.

In [ ]:
with torch.no_grad():
    cnn.eval()
    every = torch.arange(0, len(test_x), max(1, len(test_x) // 120))[:120]   # a spread of the test photos
    guess = cnn(prepare(test_x[every], 64)).argmax(1).cpu()
some = list(range(0, len(guess), max(1, len(guess) // 8)))[:8]
show(test_x[every][some], [GESTURES[guess[i]] for i in some],
     ["black" if guess[i] == test_y[every][i] else "red" for i in some])

**Now with only 20 photos of each gesture.** Most real projects have far fewer photos than 2,500. Watch what happens:

In [ ]:
few_x, few_y = pick(train_x, train_y, per_gesture=20)        # the same 60 photos for everyone
small = train(make_cnn(), few_x, few_y, rounds=15, size=64, batch=16)
scratch_few = accuracy(small, test_x, test_y, 64)
print(f"From scratch with 60 photos: {scratch_few:.1f}%   (with all {len(train_x)}: {scratch_all:.1f}%)")

Much worse: 60 photos aren't enough to learn good filters from zero. ✅ **Back to the page** for the fix: transfer learning.

## Part 3 · Borrow a trained network, then the 🏆 leaderboard

**ResNet-18** learned from 1.2 million photos of 1,000 kinds of things. We keep its layers (🔒 frozen) and swap the last one for a new layer with 3 outputs. Then we train it on the **same 60 photos**.

In [ ]:
from torchvision import models

if "few_x" not in globals():                      # jumping in here? load the photos first
    (train_x, train_y), (test_x, test_y) = get_hand_photos()
    few_x, few_y = pick(train_x, train_y, per_gesture=20)

def make_pretrained(name="resnet18", unlock=False):
    if name == "resnet18":
        net = models.resnet18(weights=models.ResNet18_Weights.DEFAULT); last = net.layer4
    elif name == "mobilenet":
        net = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT); last = net.features[-4:]
    elif name == "efficientnet":
        net = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT); last = net.features[-3:]
    else:
        raise ValueError('model_name must be "resnet18", "mobilenet" or "efficientnet"')
    for p in net.parameters():
        p.requires_grad = False                   # 🔒 keep what it learned
    if unlock:
        for p in last.parameters():
            p.requires_grad = True                # ✏️ let the last block learn too
    if name == "resnet18":
        net.fc = nn.Linear(net.fc.in_features, 3)                          # ✏️ a new last layer
    else:
        net.classifier[-1] = nn.Linear(net.classifier[-1].in_features, 3)
    return net.to(device)

tuned = train(make_pretrained(), few_x, few_y, rounds=10, size=SIZE, batch=16)
tuned_score = accuracy(tuned, test_x, test_y, SIZE)
print(f"Fine-tuned with 60 photos: {tuned_score:.1f}%")
if "scratch_few" in globals():
    print(f"From scratch with the same 60: {scratch_few:.1f}%")

Same 60 photos, far better. That's transfer learning.

### 🏆 The challenge

Teams of 3 or 4, **15 minutes**. Still only the same 60 photos: get the best test score you can. Change **one thing at a time**, then send your score to the board on screen. Send as often as you like: your best score counts.

**Two prizes:** the best score, and the **biggest jump** from your first score to your best.

| Idea | What to change |
|---|---|
| Another network | `model_name = "mobilenet"` or `"efficientnet"` |
| Unlock | `unlock = True`: the last block of the network learns too |
| Flip | `flip = True`: mirrored copies while training (**data augmentation**) |
| Longer | `rounds = 20` |
| Rate | `lr = 0.0003` or `lr = 0.003` |

First, your team name:

In [ ]:
TEAM = "Your team name"     # ✏️ the same name every time you send

In [ ]:
# ✏️ Change these
model_name, unlock, flip, rounds, lr = "resnet18", False, False, 10, 0.001

mine = train(make_pretrained(model_name, unlock), few_x, few_y, rounds, lr, size=SIZE, flip=flip, batch=16)
print(f"Your score: {accuracy(mine, test_x, test_y, SIZE):.1f}%")

### 📤 Send your score

The code is on the leaderboard on screen. Every team is scored on the same 372 test photos, so it's fair.

In [ ]:
import requests

CODE = ""   # ✏️ the code on the screen

score = round(accuracy(mine, test_x, test_y, SIZE), 1)
changed = f"{model_name}, unlock={unlock}, flip={flip}, rounds={rounds}, lr={lr}"

if TEAM.strip() in ("", "Your team name"):
    print("✏️ Type your team name in the TEAM cell above, run it, then run this cell again.")
elif not CODE.strip():
    print("✏️ Type the code from the screen in CODE, then run this cell again.")
else:
    try:
        reply = requests.post("https://gradlabs.in/api/leaderboard/deep-learning",
                              json={"code": CODE, "team": TEAM, "score": score, "changed": changed}, timeout=20)
        answer = reply.json()
    except Exception:
        answer = {"error": "Couldn't reach the leaderboard. Check the internet and try again."}
    if answer.get("ok"):
        print(f"✓ On the board: {answer['team']}, {answer['score']}%")
    else:
        print("Not sent:", answer.get("error", "something went wrong"))

## Part 4 · Play against it

An app that reads your hand and always plays the move that beats it. If it reads your hand wrong, **you** win.

It learned from computer-drawn hands on white backgrounds, so hold your hand up against a plain, light wall. `share=True` prints a link that works on your phone too, for about 3 days.

In [ ]:
%pip install -q gradio

In [ ]:
import gradio as gr

player = globals().get("mine", globals().get("tuned"))
if player is None:
    raise RuntimeError("Run Part 3 first, to train a network to play with.")
BEATS = {"rock": "paper", "paper": "scissors", "scissors": "rock"}

def model_scores(image):
    x = torch.from_numpy(np.array(image.convert("RGB").resize((SIZE, SIZE)))).permute(2, 0, 1).unsqueeze(0)
    player.eval()
    with torch.no_grad():
        return player(prepare(x, SIZE)).softmax(1)[0].cpu().tolist()

def play(image):
    if image is None:
        return "Show your hand, then press Submit.", None
    chances = model_scores(image)
    you = GESTURES[int(np.argmax(chances))]
    result = f"I see {you} ({100 * max(chances):.0f}% sure), so I play {BEATS[you]}. I win! 😎  (Unless I read your hand wrong.)"
    return result, {g: c for g, c in zip(GESTURES, chances)}

gr.Interface(play, gr.Image(type="pil", sources=["webcam", "upload"]),
             [gr.Textbox(label="Result"), gr.Label(label="What it sees")],
             title="Rock, paper, scissors vs. your network").launch(share=True)

## Bonus · Zero-shot with CLIP (a demo)

CLIP learned from hundreds of millions of photos with captions. We give it each test photo and three sentences, and it picks the one that fits best. **No training at all.** The first run downloads the model (about 600 MB).

In [ ]:
%pip install -q transformers

In [ ]:
from transformers import CLIPModel, CLIPProcessor

if "test_x" not in globals():
    (train_x, train_y), (test_x, test_y) = get_hand_photos()

clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
sentences = ["a photo of a flat open hand", "a photo of a fist", "a photo of a hand showing two fingers"]   # paper, rock, scissors

pictures = [Image.fromarray(p.permute(1, 2, 0).numpy()) for p in test_x]
chances = []
for i in range(0, len(pictures), 64):
    inputs = processor(text=sentences, images=pictures[i:i + 64], return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        chances.append(clip(**inputs).logits_per_image.softmax(1).cpu())
chances = torch.cat(chances)
print(f"CLIP, with no training: {100 * (chances.argmax(1) == test_y).float().mean():.1f}% right")

✏️ Change the `sentences` and run it again: the words you choose change the score.

## ✅ Done

You trained a neuron, a CNN, fine-tuned a pretrained network, and played against it. The page stays up: **https://gradlabs.in/learn/deep-learning/**